<a href="https://colab.research.google.com/github/evgenykomarov/sdc_course/blob/spring2026/seminar06-prediction-planning/ysda_homework_ppo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scenario Data Loading

This tutorial demonstrates how to load scenario data from the Waymo Open Motion Dataset (WOMD) using the Waymax dataloader.

In [ ]:
!pip install --upgrade pip
!pip install git+https://github.com/waymo-research/waymax.git@main#egg=waymo-waymax
!pip install matplotlib==3.8.0

Необходимые для семинара данные и код лежат по ссылке https://drive.google.com/drive/folders/1iI_1PIFNx6-5MUIQkjslg1MqRhP7mikM?usp=sharing

Необходимо создать ярлык на своем Google Drive, как в прошлой дз, если вы еще этого не сделали

Монтируем гугл диск в локальную файловую систему

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Ниже нужно указать путь до созданного ранее ярлыка, например, если ярлык имеет путь `ai360`, то должно получиться
```
SEMINAR_PATH = '/content/drive/MyDrive/ai360'
```

In [ ]:
SEMINAR_PATH = '/content/drive/MyDrive/ysda-prediction'

In [ ]:
import os
import shutil

if not os.path.exists('lib'):
    shutil.copytree(os.path.join(SEMINAR_PATH, 'lib'), 'lib')
else:
    print('"lib" folder already exists. If you want to rewrite lib by original folder, remove local "lib" manually')

In [ ]:
%%capture

import os
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


from waymax.config import DatasetConfig

## you need to import code for PlanningModel and NormalizeSceneWrapper classes from your previous homework
from <your path to code with PlanningModel class here> import PlanningModel, NormalizeSceneWrapper
from lib.data_utils import WaymaxDataset, scenario_to_features_gt

device = 'cuda'

## Здесь вам нужно загрузить веса своей обученной модели планера из предыдущего дз

In [ ]:
"""your code here"""

checkpoint = torch.load("path to your checkpoint", map_location='cuda')
model_state_dict = {k.replace('model.model.', 'model.'): v for k, v in checkpoint['state_dict'].items()
                    if k.startswith('model.model')}
# Then load into your standalone model
model = NormalizeSceneWrapper(PlanningModel())
model.load_state_dict(model_state_dict)

In [ ]:
def get_data_config(split_name, seminar_path=SEMINAR_PATH):
    split_path = os.path.join(SEMINAR_PATH, 'data', split_name)

    obj_count = int(os.listdir(split_path)[0].rsplit('-')[-1])
    return DatasetConfig(
        path=os.path.join(split_path, f'{split_name}_tfexample.tfrecord@{obj_count}'),
        max_num_objects=24,
        batch_dims=[8],
        repeat=1,
        shuffle_buffer_size=64,
        deterministic=False,
        num_shards=1
    )

In [ ]:
train_dataset = WaymaxDataset(get_data_config('training'))
val_dataset = WaymaxDataset(get_data_config('validation'))

In [ ]:
FUTURE_STEPS = 30

# Дообучение модели планнера с помощью RL

В данном домашнем задании вам предстоит реализовать алгоритм PPO - Proximal Policy Optimization для дообучения модели планнинга. Для того чтобы ознакомиться с идеей алгоритма, рекомендую прочесть полную статью https://arxiv.org/pdf/1707.06347 или краткую сводку с всей нужной информацией https://spinningup.openai.com/en/latest/algorithms/ppo.html

In [ ]:
from waymax import config as _config
from waymax import dataloader
from waymax import datatypes
from waymax import dynamics
from waymax import env as _env
from waymax import agents
from waymax import visualization
from waymax.agents import SimAgentActor, WaymaxActorOutput
from waymax import metrics

from torch.distributions import Categorical

import jax
from jax import numpy as jnp
from jax import random

import chex

import dataclasses

from typing import Any, Optional, Callable, Sequence
import mediapy

# PPO Algorithm

## Initializing the ego-agent as RL agent

**Класс EgoAgent** является классом, который управляет эго-агентом в симуляции. Он наследуется от класса SimAgentActor из Waymax и расширяет его функциональность для работы с моделью планнера, которая предсказывает следующие действия агента.


1. Имплементация метода select_action не отличается от базовой имплементации, кроме того, что мы добавляем поле log_probs в выход, чтобы во время обучения использовать их для подсчета ppo loss.


2. Метод update_trajectory является ключевым в классе EgoAgent. Он отвечает за выбор действия на основе текущей политики и возвращает логарифмические вероятности действий, которые используются для вычисления лосса в алгоритме PPO.

In [ ]:
def extract_best_mode_from_pred_component(pred, best_mode):
    return pred[torch.arange(best_mode.shape[0]), best_mode]

def extract_first_timestep_from_pred(pred):
    return pred[..., 0]

In [ ]:
ActorState = datatypes.PyTree
Params = datatypes.PyTree
Action = datatypes.PyTree


@chex.dataclass(frozen=True)
class RLActorOutput(agents.WaymaxActorOutput):
    """Output of the RL actor, extending WaymaxActorOutput with log_probs.

    Attributes:
        actor_state: Internal state for whatever the agent needs to keep as its
          state. This can be recurrent embeddings or accounting information.
        action: Action of shape (..., num_objects) predicted by the RL actor.
        is_controlled: A binary indicator of shape (..., num_objects) representing
          which objects are controlled by the actor.
        log_probs: Log_probs output by the RL actor, representing the log probability
          of policy's actions
    """

    log_probs: Optional[jax.Array]


class EgoAgent(SimAgentActor):
    def __init__(self,  model, is_controlled_func: Optional[Callable[[datatypes.SimulatorState], jax.Array]] = None):
        super().__init__(is_controlled_func=is_controlled_func)
        self.model = model


    def update_trajectory(
        self, state: datatypes.SimulatorState
    ) -> datatypes.TrajectoryUpdate:
        """Updates the trajectory for all simulated agents."""

        features, _ = scenario_to_features_gt(state, features_first_timestamp=state.timestep - 10, gt_timestamps=1)
        pred = self.model(features)
        logits = pred['logits']
        mode_distribution = Categorical(logits=logits)
        mode_sample = mode_distribution.sample()
        log_probs = mode_distribution.log_prob(mode_sample)

        trajectory_jax = jax.tree_util.tree_map(
            lambda x: jnp.repeat(
                jnp.array(
                    extract_first_timestep_from_pred(
                        extract_best_mode_from_pred_component(x, mode_sample)
                    ).cpu().detach().numpy()
                ).reshape(-1, 1, 1),
                state.log_trajectory.x.shape[-2],
                axis=-2
            ),
            pred['trajectory']
        )

        x = trajectory_jax['x']
        y = trajectory_jax['y']
        vel_x = trajectory_jax['vel_x']
        vel_y = trajectory_jax['vel_y']
        yaw = trajectory_jax['yaw']

        # predictions is valid only for sdc
        valid = jnp.bool_(jnp.zeros_like(x))
        valid = valid.at[state.object_metadata.is_sdc].set(True)

        return datatypes.TrajectoryUpdate(x=x, y=y, yaw=yaw, vel_x=vel_x, vel_y=vel_y, valid=valid), log_probs


    def select_action(
        self,
        params: Params,
        state: datatypes.SimulatorState,
        actor_state: Any,
        rng: jax.Array,
    ) -> agents.WaymaxActorOutput:
        """Selects action and updates trajectory given the current simulator state."""

        del actor_state, rng  # Not used
        action, log_probs = self.update_trajectory(state)
        action = action.as_action() # here we transform the action which we got from the model to a desired datatype [look above]

        return RLActorOutput(
            action=action,
            actor_state=None,
            is_controlled=self.is_controlled_func(state),
            log_probs=log_probs,
        )

    @property
    def name(self) -> str:
        return self.__class__.__name__

## Rewards, Reward-to-go computation and PPO Loss function

### Rewards {1.5 балла}

Ниже вам нужно будет создать различные функции для подсчета наград при помощи модуля waymax.rewards. Эти функции наград используются в симуляции для оценки поведения агентов в зависимости от их действий. Зайдите в репозиторий https://github.com/waymo-research/waymax/tree/main/waymax/metrics и поймите, как добавлять следующие реворды в обучение: imitation reward, offroad reward, overlap reward, comfort reward. Также, разберитесь в смысле каждого реворда, исходя из кода waymax/metrics, и опишите их здесь:

1. **Log divergence:** < your explanation >

2. **Offroad reward:** < your explanation >

3. **Overlap reward:** < your explanation >

4. **Comfort reward:** < your explanation >

С помощью linear combination reward из модуля waymax мы можем совмещать несколько наград и считать суммарную награду на каждом шаге симуляции. Давайте для начала добавим log_divergence, offroad, overlap rewards

В конфиге награды можно менять величину штрафа. Например, если мы видим, что модель склонна часто выезжать за пределы дороги, можно увеличить штраф за offroad, чтобы модель с большей вероятностью отвергала действия, приводящие к выезду за пределы дороги.

Также, посмотрев в код наград, подумайте, с каким знаком их нужно добавлять в наше обучение. Здесь важно помнить, что алгоритмы RL максимизируют награду

In [ ]:
from waymax.rewards import linear_combination_reward

"""
your code here
"""

imitation_config = _config.LinearCombinationRewardConfig({})
imitation_reward_function = linear_combination_reward.LinearCombinationReward(imitation_config)

offroad_config = _config.LinearCombinationRewardConfig({})
offroad_reward_function = linear_combination_reward.LinearCombinationReward(offroad_config)

overlap_config = _config.LinearCombinationRewardConfig({})
overlap_reward_function = linear_combination_reward.LinearCombinationReward(overlap_config)

all_rewards_config = _config.LinearCombinationRewardConfig({})
combination_reward_function = linear_combination_reward.LinearCombinationReward(all_rewards_config)

#### Задания 1.1 и 1.2 {2 балла}:

1.1 **Имплементировать PPO-Clip Loss** {1.5 баллов}
PPO-Clip Loss используется для оптимизации политики агента. PPO-Clip обновляет веса с помощью градиентного подъема:
$$
\theta_{k+1} = \arg \max_{\theta} \mathbb{E}_{s,a \sim \pi_{\theta_k}} \left[ L(s, a, \theta_k, \theta) \right],
$$
L имеет вид:
$$
L(s, a, \theta_k, \theta) = \min\left(
    \frac{\pi_\theta(a|s)}{\pi_{\theta_k}(a|s)} A^{\pi_{\theta_k}}(s, a), \;
    \text{clip}\left(
        \frac{\pi_\theta(a|s)}{\pi_{\theta_k}(a|s)}, 1 - \epsilon, 1 + \epsilon
    \right) A^{\pi_{\theta_k}}(s, a)
\right),
$$

Epsilon - это clipping factor, контролирующий, насколько далеко от референсной политики $\pi_{\theta_k}$ может уйти обучаемая политика $\pi_\theta$.
Так как современные оптимайзеры решают задачу минимизации, а в PPO мы наоборот максимизируем ожидаемую награду, то при обучении будем домножать лосс на -1, чтобы градиентный спуск фактически поднимал reward.

В стандартном случае, $A(s, a) = Q(s, a) - V(s)$, где $V(s)$ - произвольная функция или модель, которая умеет оценивать бейзлайны, то есть среднюю награду, которую получит наш агент, начав в стейте s и придерживаясь политики $\pi_\theta$ до конца эпизода. Основная цель добавления $V(s)$ в подсчет advantage-ей - снизить дисперсию в оценке реворда. Для того чтобы не нагромождать эту дз, мы не будем обучать отдельную Value Model для оценки бейзлайнов, вместо этого будем считать, что $A(s, a) = normalized (Q(s, a))$, что так же позволяет снизить дисперсию.

1.2  **Имплементировать Reward-to-go** {0.5 балл}


$Q(s, a)$ - функция, оценивающая ожидаемый куммулятивный реворд от того, что агент принял action a в стейте s. Есть различные способы оценивать $Q(s, a)$, например - мгновенная награда от выполнения action a или сумма всех наград за эпизод. Чтобы оценить полезность action a в state s, агент должен ориентироваться на последствия от предпринятого action-a. Реворды, которые были получены до этого момента не показывают, насколько хорошее был действие. Соответственно, мы хотим считать $Q(s, a)$ как куммулятивную сумму будущих ревордов (reward-to-go)


Формула для Reward-to-Go:
$$ G_t = \sum_{t'=t}^{T} \gamma^{t'-t} \cdot r_{t'} $$
Где: $ r_{t'} $ — награда на шаге t′. $γ$ — коэффициент дисконтирования.



In [ ]:
def compute_PPO_loss(advantages, ratio, clip_coef=0.1):
    """
    Implementation of ppo loss
    input:
    advantages.shape (bs, num_ticks)
    ratio.shape (bs, num_ticks)

    output:
    ppo_loss.shape (bs)
    """

    pass

def reward2go(reward, gamma=0.99):
    """
    Compute reward-to-go
    input:
    reward.shape (bs, num_ticks)
    gamma - float
    output:
    rtgs.shape (bs, num_ticks)
    """

    pass



## Initializing the environment and actors around us

#### Задание 2.1 {1 балл}:
В объекте state.object_metadata содержится информация о всех агентах на сцене.
**Найдите в этом объекте маску, которая возвращает True для агентов, соответствующих беспилотному автомобилю (SDC).**

**Настройте управление агентами:**
Для актора IDM_actors настройте функцию is_controlled_func так, чтобы она возвращала True для всех агентов, которые не являются беспилотным автомобилем.
Для актора actor_ego настройте функцию is_controlled_func так, чтобы она возвращала True только для агентов, которые являются беспилотным автомобилем.

In [ ]:
dynamics_model = dynamics.StateDynamics()

# Number of agents on the scene, can be changed if you have enough compute to take actions for more agents in your env
max_num_objects = 12

env = _env.MultiAgentEnvironment(
    dynamics_model=dynamics_model,
    config=dataclasses.replace(
        _config.EnvironmentConfig(),
        max_num_objects=max_num_objects,
        controlled_object=_config.ObjectType.VALID,
    ),
)

In [ ]:
IDM_actors = agents.IDMRoutePolicy(
    is_controlled_func=lambda state: ""your code here""
)  # these are intelligent driver models, see more here https://github.com/waymo-research/waymax/blob/main/waymax/agents/waypoint_following_agent.py#L200

actor_ego = EgoAgent(model, is_controlled_func=lambda state: ""your code here"")

actors = [actor_ego, IDM_actors]

select_action_list = [actor.select_action for actor in actors]
step = env.step

# Train loop for PPO {5 баллов}

В данном задании вам предстоит заполнить пропуски в train loop-е для обучения алгоритма PPO.

In [ ]:
def train_one_batch_ppo(scenario, start_timestep, gamma=1, clip_epsilon=0.2, ppo_epochs=3):

    rollout_data = {
        'states': [],
        'actions': [],
        'log_probs': [],
        'rewards': [],
        'masks': [],
    }

    current_state = env.reset(scenario)
    for timestep in range(start_timestep, start_timestep + FUTURE_STEPS):

        # main cycle, log_probs from reference policy don't require gradients, they can be perceived as constants
        with torch.no_grad():
            outputs = [
                select_action({'timestep': timestep}, current_state, None, None)
                for select_action in select_action_list
            ]
            action = agents.merge_actions(outputs)
            log_probs = outputs[0].log_probs.to(device)

        agent_mask = current_state.object_metadata.is_sdc

        # reward computing
        total_reward = combination_reward_function.compute(current_state, action, agent_mask).mean(axis=-1)
        reward = torch.tensor(np.asarray(total_reward), device=device)

        rollout_data['states'].append(current_state)
        rollout_data['log_probs'].append(log_probs)
        rollout_data['rewards'].append(reward)
        rollout_data['masks'].append(agent_mask)

        # perform step in the environment
        next_state = step(current_state, action)
        current_state = """your code here"""


    rewards = torch.stack(rollout_data['rewards'], dim=1)
    old_log_probs = torch.stack(rollout_data['log_probs'], dim=1)

    rtgs = reward2go(rewards, gamma=gamma)
    # normalize by batch reward-to-gos (this will be our advantages)
    normalized_rtgs = # your code here

    for _ in range(ppo_epochs):
        new_log_probs = []
        for timestep in range(FUTURE_STEPS):
            outputs = """your code here"""

            new_log_prob = """your code here"""
            new_log_probs.append(new_log_prob)

        new_log_probs = torch.stack(new_log_probs, dim=1)
        ratios = """your code here"""

        policy_loss = """your code here"""

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

    # Calculate metrics for logging (mean across batches)
    avg_step_reward = torch.stack(rollout_data['rewards']).mean(dim=0).mean().item()

    del rewards, log_probs, rtgs, normalized_rtgs
    torch.cuda.empty_cache()

    return {
        'loss': policy_loss.item(),
        'avg_step_reward': avg_step_reward,
    }


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=2.5e-4)
epochs = 1
gamma = 0.99
log_interval = 3
clip_epsilon = """your code here, remember what the clip_epsilon parameter stands for and search the internet on how big it should be"""
ppo_epochs = 3

for epoch in range(epochs):
    for scenario_idx, scenario in enumerate(train_dataset):
        metrics = train_one_batch_ppo(
            scenario,
            start_timestep=11,
            gamma=gamma,
            clip_epsilon=clip_epsilon,
            ppo_epochs=ppo_epochs
        )

        """
        add logic to save model checkpoint every 2-3 epochs, because google colab might accidentally crush because of long training

        your code here
        """

        if (scenario_idx + 1) % log_interval == 0:
            print(f"Epoch {epoch}, Scenario {scenario_idx + 1}:")
            print(f"  Total Loss: {metrics['loss']:.4f}")
            print(f"  Avg Step Reward: {metrics['avg_step_reward']:.4f}")
            print("-" * 40)

### Можете остановить обучение, когда будете достигать Avg Step Reward = -0.15 или когда у вас кончится квота в колабе:)

# Проверим обученную модель на валидационном датасете

Так как делать шаги в симуляторе довольно дорого и долго, мы прогоним на валидации только часть датасета, чтобы не забирать у вас всю гпу квоту. Если у вас заканчивается квота, можете уменьшить количество сценариев

In [ ]:
def evaluation(dataset, history_size=11, log_interval=1, num_scenarios=5):
    metrics = {
        'total': [],
        'distance': [],
        'offroad': [],
        'overlap': []
    }

    for i, scenario in enumerate(dataset, 1):
        episode_metrics = {k: [] for k in metrics}
        state = env.reset(scenario)

        for timestep in range(history_size, history_size + FUTURE_STEPS):
            with torch.no_grad():
                outputs = [select_action({'timestep': timestep}, state, None, None)
                         for select_action in select_action_list]
                action = agents.merge_actions(outputs)
                mask = state.object_metadata.is_sdc

            # Compute all rewards at once
            rewards = {
                'total': combination_reward_function.compute(state, action, mask),
                'distance': imitation_reward_function.compute(state, action, mask),
                'offroad': offroad_reward_function.compute(state, action, mask),
                'overlap': overlap_reward_function.compute(state, action, mask)
            }

            # Store rewards
            for k in episode_metrics:
                episode_metrics[k].append(torch.tensor(np.asarray(rewards[k]), device=device).mean(axis=-1))

            state = step(state, action)

        # Stack and store episode results
        for k in metrics:
            metrics[k].append(torch.stack(episode_metrics[k], dim=1))

        # Periodic logging
        if i % log_interval == 0:
            print(f"Validation Metrics, step: {i}")
            for k, v in metrics.items():
                mean_reward = torch.mean(torch.cat([m.mean(dim=1) for m in v]))
                print(f"Mean Episode {k.capitalize()} Reward: {mean_reward.item():.2f}")
        if i == num_scenarios:
            break

evaluation(val_dataset)

# Rollout generation - генерация видосиков {0.5 балла}
Скачайте несколько роллаутов и приложите их вместе с домашним заданием

In [ ]:
def generate_close_loop(model, scenario, steps=None):
    dynamics_model = dynamics.StateDynamics()

    max_num_objects = 12

    # Environment to control all objects on scene
    env = _env.MultiAgentEnvironment(
        dynamics_model=dynamics_model,
        config=dataclasses.replace(
            _config.EnvironmentConfig(),
            max_num_objects=max_num_objects,
            controlled_object=_config.ObjectType.VALID,
        ),
    )

    state = env.reset(scenario)

    # intelligent driver model actor for non-ego objects
    IDM_actors = agents.IDMRoutePolicy(
        is_controlled_func=lambda state: state.object_metadata.is_sdc == False
    )

    # our model actor for sdc
    actor_ego = EgoAgent(
        model,
        is_controlled_func=lambda state: state.object_metadata.is_sdc == True
    )

    actors = [actor_ego, IDM_actors]
    select_action_list = [actor.select_action for actor in actors]

    states = [env.reset(scenario)]

    if steps is None:
        steps = states[0].remaining_timesteps

    for timestep in range(0, steps):
        current_state = states[-1]
        outputs = [
            select_action({'timestep': timestep}, current_state, None, None)
            for select_action in select_action_list
        ]
        # make action for all objects on scene
        action = agents.merge_actions(outputs)

        # make step
        next_state = env.step(current_state, action)
        states.append(next_state)

    return states[1:]

In [ ]:
from collections import defaultdict
from waymax import metrics

def plot_states(states, use_log_traj=False, batch_idx=0):
    imgs = []
    for state in states:
        imgs.append(visualization.plot_simulator_state(
            state, use_log_traj=use_log_traj, batch_idx=batch_idx))
    mediapy.show_video(imgs, fps=10)

def get_ego_metrics(states, start_index=0):
    metrics_config = _config.MetricsConfig()

    metrics_per_time = defaultdict(list)
    for state in states[start_index:]:
        all_metrics = metrics.run_metrics(state, metrics_config)
        for k, v in all_metrics.items():
            metrics_per_time[k].append(np.asarray(v.value[state.object_metadata.is_sdc]))

    return {
        k: np.mean(metrics_per_time[k], axis=0)
        for k in metrics_per_time.keys()
    }

In [ ]:
scenario = next(iter(val_dataset))

In [ ]:
states = generate_close_loop(model, scenario, steps=50)
print(get_ego_metrics(states))
plot_states(states, batch_idx=0)

In [ ]:
plot_states(states, batch_idx=1)